# Step 11 — Revision, part 1: corrections, statistics, and "does h matter for sampling?"

Responds to review points 2, 3, 4, 5, 6, 9.

**A. Corrected fold criterion (GPU, short).** The fold condition is `4a^3 > 27 b h^2` (not `27h^2/b`). We drop the cubic-fit `tau_fold` entirely and recompute every network transition with the bisected zero crossing of the binned drift, the same method as for exact diagrams. Needs the Step 7 and Step 10 checkpoint outputs mounted.

**B. Statistics (CPU).** Per-pair bootstrap CIs on `h`; per-dataset pitchfork floor from a *symmetrized* copy of each pair (class b replaced by the mirror image of class a); sensitivity of `h` to the LDA regularizer λ and the fit window on real data; log–log exponent of jump vs `|h|/2b` with bootstrap CI.

**C. VP offset (CPU).** Convert the detector displacement in τ into a VP-time offset Δt = ½ ln(τ_peak/τ_bif) for all pairs.

**D. Does h affect sampling error? (CPU, all 45 pairs, exact drifts).** Integrate the exact 1-D reduced dynamics with (i) Heun at 10/20/40 steps, (ii) Euler–Maruyama stochastic reverse SDE at 40/200 steps, (iii) Heun 40 steps with injected score noise; measure class-fraction error and Wasserstein-1 error of the final `m` distribution against a 2000-step reference; correlate with `|h|`, `a` and `b`. This tests whether the fold/pitchfork distinction changes any sampler observable.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math, glob, os
import matplotlib.pyplot as plt
from itertools import combinations
from torchvision import datasets
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev)
rng = np.random.default_rng(0); plt.rcParams['figure.dpi'] = 130; SIGMA_DATA = 0.5
def load(ds_cls, root):
    d = ds_cls(root, train=True, download=True); return (d.data.numpy().astype(np.float32)/127.5 - 1.0).reshape(-1, 784), d.targets.numpy()
X_MN, Y_MN = load(datasets.MNIST, '/kaggle/working/mnist'); X_FM, Y_FM = load(datasets.FashionMNIST, '/kaggle/working/fmnist')

GRID = np.linspace(-4, 4, 4001); DG = GRID[1]-GRID[0]
def smooth_hist(m0, tau):
    h, _ = np.histogram(m0, bins=np.append(GRID-DG/2, GRID[-1]+DG/2), density=True); k = np.exp(-GRID**2/(2*tau)); k /= k.sum()
    return np.maximum(np.convolve(h, k, mode='same'), 1e-300)
def exact_drift(m0, tau):
    p = smooth_hist(m0, tau); return GRID, np.gradient(np.log(p), GRID), p
def _merge(pts, merge=0.02):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary_points(p):
    d = np.gradient(np.log(p), GRID); keep = p > 1e-4*p.max()
    idx = np.where((np.sign(d[:-1]) != np.sign(d[1:])) & keep[:-1] & keep[1:])[0]
    return _merge([(float(GRID[i]-d[i]*DG/(d[i+1]-d[i])) if d[i+1] != d[i] else float(GRID[i]), 'max' if d[i] > 0 else 'min') for i in idx])
def n_max(pts): return sum(k == 'max' for _, k in pts)
def birth_bisect(get_pts, taus=np.geomspace(0.05, 3.0, 60), iters=12):
    ts = sorted(taus, reverse=True); hi = None
    for tau in ts:
        pts = get_pts(tau)
        if n_max(pts) >= 2 and hi is not None: lo = tau; break
        if n_max(pts) == 1: hi = tau
    else: return None
    for _ in range(iters):
        mid = np.sqrt(lo*hi)
        if n_max(get_pts(mid)) >= 2: lo = mid
        else: hi = mid
    pm = [p for p, k in get_pts(hi) if k == 'max']; prev = pm[0] if pm else 0.0; pl = get_pts(lo)
    mx = sorted(p for p, k in pl if k == 'max'); mn = [p for p, k in pl if k == 'min']
    return float(lo), float(abs((mn[0] if mn else 0.5*(mx[0]+mx[-1])) - prev))
def lda_direction(Xa, Xb, lam=1.0):
    mu_a, mu_b = Xa.mean(0), Xb.mean(0); delta = mu_a-mu_b
    u = np.linalg.solve(np.cov(Xa.T)+np.cov(Xb.T)+lam*np.eye(784), delta); u /= np.linalg.norm(u)
    return u.astype(np.float32), (0.5*(mu_a+mu_b)).astype(np.float32), 0.5*float(delta @ u)
def balanced(X, Y, pair):
    Xa = X[Y == pair[0]]; Xb = X[Y == pair[1]]; n = min(len(Xa), len(Xb)); return Xa[:n], Xb[:n]
def cusp_fit(m, f, w, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(m) < window) & (w > 0); c3, c2, c1, c0 = np.polyfit(m[ok], f[ok], 3, w=np.sqrt(w[ok]))
    b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    a = c1 + 3*c3*s**2 + 2*c2*s; h = c0 + c1*s + c2*s**2 + c3*s**3
    return dict(h=h, a=a, b=b, fold=(a > 0) and (4*a**3 > 27*b*h**2))     # CORRECTED condition
def analyse_pair(Xa, Xb, lam=1.0, window=1.0):
    u, mid, half = lda_direction(Xa, Xb, lam); m0 = np.concatenate([(Xa-mid) @ u, (Xb-mid) @ u]) / half
    bj = birth_bisect(lambda t: stationary_points(smooth_hist(m0, t)))
    if bj is None: return None
    f = cusp_fit(*exact_drift(m0, bj[0]), window=window); return dict(tau=bj[0], jump=bj[1], **f, m0=m0, u=u, mid=mid, half=half)

## A. Corrected fold condition and bisected network transitions (GPU)

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, c, emb):
        super().__init__(); self.n1 = nn.GroupNorm(8, c); self.c1 = nn.Conv2d(c, c, 3, padding=1); self.n2 = nn.GroupNorm(8, c); self.c2 = nn.Conv2d(c, c, 3, padding=1); self.e = nn.Linear(emb, c)
    def forward(self, x, e):
        h = self.c1(F.silu(self.n1(x))) + self.e(e)[:, :, None, None]; return x + self.c2(F.silu(self.n2(h)))
class UNet(nn.Module):
    def __init__(self, ch=64, emb=128):
        super().__init__(); self.emb = nn.Sequential(nn.Linear(64, emb), nn.SiLU(), nn.Linear(emb, emb)); self.inp = nn.Conv2d(1, ch, 3, padding=1)
        self.d1 = ResBlock(ch, emb); self.down = nn.Conv2d(ch, 2*ch, 3, stride=2, padding=1); self.d2 = ResBlock(2*ch, emb); self.mid = ResBlock(2*ch, emb)
        self.up = nn.ConvTranspose2d(2*ch, ch, 4, stride=2, padding=1); self.u1 = ResBlock(ch, emb); self.out = nn.Conv2d(ch, 1, 3, padding=1)
    def forward(self, x, c_noise):
        f = torch.exp(torch.arange(32, device=x.device) * -(math.log(1000)/31)); e = self.emb(torch.cat([torch.sin(c_noise[:, None]*f), torch.cos(c_noise[:, None]*f)], 1))
        h1 = self.d1(self.inp(x), e); h2 = self.mid(self.d2(self.down(h1), e), e); return self.out(self.u1(self.up(h2) + h1, e))
def denoise(net, x, sigma):
    s = sigma.view(-1, 1, 1, 1); c_skip = SIGMA_DATA**2/(s**2 + SIGMA_DATA**2); c_out = s*SIGMA_DATA/torch.sqrt(s**2 + SIGMA_DATA**2)
    return c_skip*x + c_out*net(x/torch.sqrt(s**2 + SIGMA_DATA**2), 0.25*torch.log(sigma))
def zero_crossings(cen, mean, cnt, min_count=150):
    ok = cnt >= min_count; c, f = cen[ok], mean[ok]; out = []
    for i in range(len(c)-1):
        if np.sign(f[i]) != np.sign(f[i+1]) and f[i] != 0: out.append((float(c[i]-f[i]*(c[i+1]-c[i])/(f[i+1]-f[i])), 'max' if f[i] > 0 else 'min'))
    return _merge(out, merge=0.05)
@torch.no_grad()
def net_points(net, Xflat, u, mid, half, tau, n=30000, nbins=70):
    ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev); sigma = half*np.sqrt(tau)
    x0 = torch.from_numpy(Xflat[rng.choice(len(Xflat), n)]).to(dev); xt = x0 + sigma*torch.randn_like(x0)
    D = torch.cat([denoise(net, xb.view(-1,1,28,28), torch.full((len(xb),), sigma, device=dev)).view(len(xb), -1) for xb in xt.split(2000)])
    m = ((xt-midt) @ ut / half).cpu().numpy(); sm = (half*((D-xt) @ ut)/sigma**2).cpu().numpy()
    lo, hi = np.quantile(m, [0.005, 0.995]); edges = np.linspace(lo, hi, nbins+1); b = np.clip(np.digitize(m, edges)-1, 0, nbins-1)
    cnt = np.bincount(b, minlength=nbins).astype(float); s = np.bincount(b, weights=sm, minlength=nbins); k = np.exp(-0.5*(np.arange(-5, 6)/1.5)**2)
    return zero_crossings(0.5*(edges[:-1]+edges[1:]), np.convolve(s, k, 'same')/np.maximum(np.convolve(cnt, k, 'same'), 1e-9), cnt)

ck = sorted(glob.glob('/kaggle/input/**/unet_*.pt', recursive=True)); print(len(ck), 'checkpoints found')
print(f"{'checkpoint':>28} {'exact tau_bif':>14} {'network tau_bif':>16} {'net jump':>9}")
NETBIF = {}
for path in ck:
    name = os.path.basename(path).replace('.pt', '')
    tag = name.replace('unet_', ''); fm = tag.startswith('fm'); tag = tag[2:] if fm else tag
    pair = (int(tag[0]), int(tag[1])); X, Y = (X_FM, Y_FM) if fm else (X_MN, Y_MN)
    Xa, Xb = balanced(X, Y, pair); r = analyse_pair(Xa, Xb); Xflat = np.concatenate([Xa, Xb])
    net = UNet().to(dev); net.load_state_dict(torch.load(path, map_location=dev)); net.eval()
    bj = birth_bisect(lambda t: net_points(net, Xflat, r['u'], r['mid'], r['half'], t), taus=np.geomspace(0.1, 3.0, 36))
    NETBIF[name] = (r['tau'], bj); print(f"{name:>28} {r['tau']:14.3f} {(bj[0] if bj else float('nan')):16.3f} {(bj[1] if bj else float('nan')):9.2f}")

## B. Statistics: bootstrap CIs, per-dataset floor, λ and window sensitivity, exponent

In [ ]:
def census(X, Y, name, lam=1.0, window=1.0, nboot=0):
    out = []
    for a_, b_ in combinations(range(10), 2):
        Xa, Xb = balanced(X, Y, (a_, b_)); r = analyse_pair(Xa, Xb, lam, window)
        if r is None: continue
        rec = dict(pair=(a_, b_), tau=r['tau'], jump=r['jump'], h=r['h'], a=r['a'], b=r['b'])
        if nboot:
            hs = []
            for _ in range(nboot):
                ia = rng.choice(len(Xa), len(Xa)); ib = rng.choice(len(Xb), len(Xb)); rb = analyse_pair(Xa[ia], Xb[ib], lam, window)
                if rb: hs.append(rb['h'])
            rec['h_lo'], rec['h_hi'] = np.percentile(hs, [2.5, 97.5])
        out.append(rec)
    return out

def floor_from_symmetrized(X, Y, name, npairs=10):
    """Per-dataset pitchfork floor: class b is replaced by the mirror image of an independent half of class a, so the two 'classes'
    are symmetric in distribution but differ by sampling noise, exactly as a true pitchfork pair would. The 95th percentile of |h| is the floor."""
    hs = []
    for a_, b_ in list(combinations(range(10), 2))[:npairs]:
        Xa, Xb = balanced(X, Y, (a_, b_)); u, mid, half = lda_direction(Xa, Xb)
        pa = (Xa-mid) @ u / half; idx = rng.permutation(len(pa)); k = len(pa)//2
        m0 = np.concatenate([pa[idx[:k]], -pa[idx[k:2*k]]])                 # mirror an INDEPENDENT half: symmetric in distribution, real sampling noise
        bj = birth_bisect(lambda t: stationary_points(smooth_hist(m0, t)))
        if bj: hs.append(abs(cusp_fit(*exact_drift(m0, bj[0]))['h']))
    return np.array(hs)

for X, Y, name in [(X_MN, Y_MN, 'MNIST'), (X_FM, Y_FM, 'FashionMNIST')]:
    fl = floor_from_symmetrized(X, Y, name)
    print(f"{name}: symmetrized-control |h| floor: median {np.median(fl):.4f}, 95th pct {np.percentile(fl, 95):.4f}")
    main = census(X, Y, name, nboot=40)
    H = np.array([abs(r['h']) for r in main]); lo = np.array([r['h_lo'] for r in main]); hi = np.array([r['h_hi'] for r in main])
    above = sum((abs(r['h']) > np.percentile(fl, 95)) and (r['h_lo'] > 0 or r['h_hi'] < 0) for r in main)
    print(f"  {len(main)} pairs; |h| median {np.median(H):.3f}; pairs with |h| above floor AND bootstrap CI excluding 0: {above}/{len(main)}")
    for lam in [0.1, 10.0]:
        alt = census(X, Y, name, lam=lam); hh = {r['pair']: r['h'] for r in alt}
        print(f"  lambda={lam}: corr with lambda=1 h over pairs = {np.corrcoef([r['h'] for r in main if r['pair'] in hh], [hh[r['pair']] for r in main if r['pair'] in hh])[0,1]:.3f}")
    for win in [0.8, 1.2]:
        alt = census(X, Y, name, window=win); hh = {r['pair']: r['h'] for r in alt}
        print(f"  window={win}: median |h_win - h_1.0| = {np.median([abs(r['h']-hh[r['pair']]) for r in main if r['pair'] in hh]):.4f}")
    # exponent with CI (exclude floor pairs)
    ok = H > np.percentile(fl, 95); x = np.log(H[ok]/(2*np.array([r['b'] for r in main])[ok])); y = np.log(np.array([r['jump'] for r in main])[ok])
    sl, ic = np.polyfit(x, y, 1); B = [np.polyfit(*(lambda k: (x[k], y[k]))(rng.choice(len(x), len(x))), 1)[0] for _ in range(2000)]
    print(f"  jump = C (|h|/2b)^p: p = {sl:.3f} [95% CI {np.percentile(B,2.5):.3f}, {np.percentile(B,97.5):.3f}], C = {np.exp(ic):.2f} (theory p=1/3, C=3); n={ok.sum()}")
    if name == 'MNIST': MN = main; MN_FLOOR = np.percentile(fl, 95)
    else: FM = main; FM_FLOOR = np.percentile(fl, 95)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for k, (main, fl, name) in enumerate([(MN, MN_FLOOR, 'MNIST'), (FM, FM_FLOOR, 'FashionMNIST')]):
    o = np.argsort([abs(r['h']) for r in main])
    hv = np.array([main[i]['h'] for i in o]); lo_ = np.array([main[i]['h_lo'] for i in o]); hi_ = np.array([main[i]['h_hi'] for i in o])
    ax[k].errorbar(range(len(o)), hv, yerr=[np.maximum(hv-lo_, 0), np.maximum(hi_-hv, 0)], fmt='o', ms=3)
    ax[k].axhspan(-fl, fl, color='k', alpha=0.12, label='per-dataset floor (95th pct of symmetrized control)'); ax[k].axhline(0, color='k', lw=0.5)
    ax[k].set_title(f'{name}: signed h per pair with bootstrap CI'); ax[k].set_xlabel('pair (sorted by |h|)'); ax[k].legend(fontsize=7)
plt.tight_layout(); plt.show()

## C. Detector displacement as a VP-time offset

In [ ]:
def varH_peak(Xa, Xb, r, taus=np.geomspace(0.05, 5.0, 60), n=60000):
    A = (Xa - r['mid']) @ r['u'] / r['half']; B = (Xb - r['mid']) @ r['u'] / r['half']; out = []
    for tau in taus:
        pa, pb = smooth_hist(A, tau), smooth_hist(B, tau)
        m = np.concatenate([A[rng.choice(len(A), n//2)], B[rng.choice(len(B), n//2)]]) + np.sqrt(tau)*rng.standard_normal(n)
        ia = np.clip(((m-GRID[0])/DG).round().astype(int), 0, len(GRID)-1); pc = np.clip(pa[ia]/(pa[ia]+pb[ia]), 1e-12, 1-1e-12)
        out.append((-(pc*np.log(pc)+(1-pc)*np.log(1-pc))).var())
    return taus[int(np.argmax(out))]
ratios, dts = [], []
for r in MN:
    Xa, Xb = balanced(X_MN, Y_MN, r['pair']); full = analyse_pair(Xa, Xb); tp = varH_peak(Xa, Xb, full)
    ratios.append(tp/r['tau']); dts.append(0.5*np.log(tp/r['tau']))
print(f"MNIST: tau_peak/tau_bif median {np.median(ratios):.2f} (range {min(ratios):.2f}-{max(ratios):.2f});  VP-time offset dt = 0.5 ln(ratio): median {np.median(dts):+.2f} (range {min(dts):+.2f} to {max(dts):+.2f})")
print("Interpretation: the variance peak sits at t_S + O(1) as the Information Dynamics theorem allows; this is the size and sign of that O(1).")

## D. Does h change any sampler observable? (exact 1-D reduced dynamics, all 45 pairs)

Reduced dynamics `dm/dσ = −σ g(m;σ)/half²` with the exact drift `g`. Reference: Heun, 400 steps (converged in 1-D). Observables: class-fraction error and W1 distance of the final `m` distribution to the reference, for (i) Heun 10/20/40 steps, (ii) reverse SDE (Euler–Maruyama, EDM noise) 40/200 steps, (iii) Heun 40 with i.i.d. Gaussian perturbation of the drift of relative size 0.3.

In [ ]:
from scipy.signal import fftconvolve
def sig_sched(steps, s_max=20.0, s_min=0.002, rho=7.0):
    i = np.arange(steps); return (s_max**(1/rho) + i/(steps-1)*(s_min**(1/rho) - s_max**(1/rho)))**rho
def make_g(m0):
    cache = {}
    def g(m, sigma, half, noise=0.0):
        key = round(float(sigma), 6)
        if key not in cache:
            tau = sigma**2/half**2; c = np.linspace(-30, 30, 12001)
            p = np.histogram(m0, bins=np.append(c-(c[1]-c[0])/2, c[-1]+(c[1]-c[0])/2), density=True)[0]
            k = np.exp(-c**2/(2*tau)); k /= k.sum(); p = np.maximum(fftconvolve(p, k, 'same'), 1e-300); cache[key] = (c, np.gradient(np.log(p), c))
        c, gg = cache[key]; val = np.interp(m, c, gg)
        return val*(1 + noise*rng.standard_normal(len(m))) if noise else val
    return g
def run(g, half, steps, kind='heun', noise=0.0, n=20000, seed=1):
    r = np.random.default_rng(seed); s = sig_sched(steps); m = r.standard_normal(n)*s[0]/half - 0.0
    for k in range(steps-1):
        s0, s1 = s[k], s[k+1]
        if kind == 'heun':
            d0 = -s0*g(m, s0, half, noise)/half**2; m1 = m + (s1-s0)*d0; d1 = -s1*g(m1, s1, half, noise)/half**2; m = m + (s1-s0)*0.5*(d0+d1)
        else:   # reverse SDE, Euler-Maruyama: dm = -2 sigma g/half^2 dsigma + sqrt(2 sigma |dsigma|)/half dW
            m = m + (s1-s0)*(-2*s0*g(m, s0, half, noise)/half**2) + np.sqrt(2*s0*abs(s1-s0))/half*r.standard_normal(n)
    return m
def w1(a, b): a = np.sort(a); b = np.sort(b); q = np.linspace(0, 1, 2001)[1:-1]; return float(np.mean(np.abs(np.quantile(a, q)-np.quantile(b, q))))

rows = []
for r in MN:
    Xa, Xb = balanced(X_MN, Y_MN, r['pair']); full = analyse_pair(Xa, Xb); g = make_g(full['m0']); half = full['half']
    ref = run(g, half, 400); fr = (ref > 0).mean(); rec = dict(pair=r['pair'], h=abs(r['h']), a=r['a'], b=r['b'])
    for nm, kw in [('heun10', dict(steps=10)), ('heun20', dict(steps=20)), ('heun40', dict(steps=40)), ('sde40', dict(steps=40, kind='sde')), ('sde200', dict(steps=200, kind='sde')), ('noisy40', dict(steps=40, noise=0.3))]:
        m = run(g, half, **kw); rec[nm+'_dfrac'] = abs((m > 0).mean()-fr); rec[nm+'_w1'] = w1(m, ref)
    rows.append(rec); print(r['pair'], {k: round(v, 3) for k, v in rec.items() if k not in ('pair',)})
H = np.array([x['h'] for x in rows]); ok = H > MN_FLOOR
print("\nSpearman correlation of each sampler error with |h| across pairs (floor pairs excluded):")
from scipy.stats import spearmanr
for nm in ['heun10', 'heun20', 'heun40', 'sde40', 'sde200', 'noisy40']:
    for q in ['dfrac', 'w1']:
        v = np.array([x[nm+'_'+q] for x in rows]); print(f"  {nm:>8} {q:>5}: rho = {spearmanr(H[ok], v[ok]).correlation:+.2f}   (median error {np.median(v):.3f})")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for nm, c in [('heun10', 'C0'), ('sde40', 'C1'), ('noisy40', 'C2')]:
    ax[0].scatter(H, [x[nm+'_dfrac'] for x in rows], s=14, color=c, label=nm); ax[1].scatter(H, [x[nm+'_w1'] for x in rows], s=14, color=c, label=nm)
for a_ in ax: a_.set_xscale('log'); a_.set_xlabel('|h|'); a_.legend(fontsize=7)
ax[0].set_ylabel('|class fraction - reference|'); ax[1].set_ylabel('W1(final m, reference)'); plt.tight_layout(); plt.show()

## What to report back
**A.** The table of bisected network transitions per checkpoint (replaces every `tau_fold` in the paper).
**B.** Per-dataset floor, the "above floor AND CI excludes zero" counts, λ and window sensitivity lines, and the exponent with CI.
**C.** The VP-offset line.
**D.** The Spearman table. This decides whether the paper can claim the fold/pitchfork distinction matters for samplers, or must say it does not at these |h|.